In [ ]:
from google.colab import files
uploaded = files.upload()


Saving saleshourly.csv to saleshourly.csv


In [ ]:
import os
os.listdir()


['.config', 'saleshourly.csv', 'sample_data']

In [ ]:
import pandas as pd
df = pd.read_csv("saleshourly.csv")
df.head()


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name
0,1/2/2014 8:00,0.0,0.67,0.4,2.0,0.0,0.0,0.0,1.0,2014,1,8,Thursday
1,1/2/2014 9:00,0.0,0.00,1.0,0.0,2.0,0.0,0.0,0.0,2014,1,9,Thursday
2,1/2/2014 10:00,0.0,0.00,0.0,3.0,2.0,0.0,0.0,0.0,2014,1,10,Thursday
3,1/2/2014 11:00,0.0,0.00,0.0,2.0,1.0,0.0,0.0,0.0,2014,1,11,Thursday
4,1/2/2014 12:00,0.0,2.00,0.0,5.0,2.0,0.0,0.0,0.0,2014,1,12,Thursday


In [ ]:
df.shape


(50532, 13)

In [ ]:
df.columns


Index(['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03',
       'R06', 'Year', 'Month', 'Hour', 'Weekday Name'],
      dtype='object')

In [ ]:
df["datum"] = pd.to_datetime(df["datum"])

df["Date"] = df["datum"].dt.date
df["Year"] = df["datum"].dt.year
df["Month"] = df["datum"].dt.month
df["Hour"] = df["datum"].dt.hour
df["Weekday"] = df["datum"].dt.day_name()

df.head()


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name,Date,Weekday
0,2014-01-02 08:00:00,0.0,0.67,0.4,2.0,0.0,0.0,0.0,1.0,2014,1,8,Thursday,2014-01-02,Thursday
1,2014-01-02 09:00:00,0.0,0.00,1.0,0.0,2.0,0.0,0.0,0.0,2014,1,9,Thursday,2014-01-02,Thursday
2,2014-01-02 10:00:00,0.0,0.00,0.0,3.0,2.0,0.0,0.0,0.0,2014,1,10,Thursday,2014-01-02,Thursday
3,2014-01-02 11:00:00,0.0,0.00,0.0,2.0,1.0,0.0,0.0,0.0,2014,1,11,Thursday,2014-01-02,Thursday
4,2014-01-02 12:00:00,0.0,2.00,0.0,5.0,2.0,0.0,0.0,0.0,2014,1,12,Thursday,2014-01-02,Thursday


In [ ]:
drug_cols = ["M01AB","M01AE","N02BA","N02BE","N05B","N05C","R03","R06"]

df_long = df.melt(
    id_vars=["datum","Date","Year","Month","Hour","Weekday"],
    value_vars=drug_cols,
    var_name="Therapy",
    value_name="Sales"
)

df_long.head()


,datum,Date,Year,Month,Hour,Weekday,Therapy,Sales
0,2014-01-02 08:00:00,2014-01-02,2014,1,8,Thursday,M01AB,0.0
1,2014-01-02 09:00:00,2014-01-02,2014,1,9,Thursday,M01AB,0.0
2,2014-01-02 10:00:00,2014-01-02,2014,1,10,Thursday,M01AB,0.0
3,2014-01-02 11:00:00,2014-01-02,2014,1,11,Thursday,M01AB,0.0
4,2014-01-02 12:00:00,2014-01-02,2014,1,12,Thursday,M01AB,0.0


In [ ]:
therapy_map = {
    "M01AB": "Anti-inflammatory (Rheumatic)",
    "M01AE": "NSAID Painkillers",
    "N02BA": "Aspirin Class",
    "N02BE": "Paracetamol",
    "N05B": "Anti-Anxiety",
    "N05C": "Sleep & Sedatives",
    "R03": "Asthma",
    "R06": "Antihistamine"
}

df_long["Therapy_Name"] = df_long["Therapy"].map(therapy_map)

df_long.head()


,datum,Date,Year,Month,Hour,Weekday,Therapy,Sales,Therapy_Name
0,2014-01-02 08:00:00,2014-01-02,2014,1,8,Thursday,M01AB,0.0,Anti-inflammatory (Rheumatic)
1,2014-01-02 09:00:00,2014-01-02,2014,1,9,Thursday,M01AB,0.0,Anti-inflammatory (Rheumatic)
2,2014-01-02 10:00:00,2014-01-02,2014,1,10,Thursday,M01AB,0.0,Anti-inflammatory (Rheumatic)
3,2014-01-02 11:00:00,2014-01-02,2014,1,11,Thursday,M01AB,0.0,Anti-inflammatory (Rheumatic)
4,2014-01-02 12:00:00,2014-01-02,2014,1,12,Thursday,M01AB,0.0,Anti-inflammatory (Rheumatic)


In [ ]:
df_long["Time_Block"] = df_long["Hour"].apply(
    lambda x: "Night" if (x >= 20 or x <= 6) else "Day"
)


In [ ]:
df_long.to_excel("pharma_clean_data.xlsx", index=False)


In [ ]:
from google.colab import files
files.download("pharma_clean_data.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
market_share = (
    df_long
    .groupby("Therapy_Name")["Sales"]
    .sum()
    .reset_index()
    .sort_values("Sales", ascending=False)
)


In [16]:
monthly_trends = (
    df_long
    .groupby(["Year", "Month", "Therapy_Name"])["Sales"]
    .sum()
    .reset_index()
)


In [17]:
day_night = (
    df_long
    .groupby(["Therapy_Name", "Time_Block"])["Sales"]
    .sum()
    .reset_index()
)


In [18]:
yearly = (
    df_long
    .groupby(["Year", "Therapy_Name"])["Sales"]
    .sum()
    .reset_index()
)


In [19]:
with pd.ExcelWriter("pharma_business_tables.xlsx") as writer:
    market_share.to_excel(writer, sheet_name="Market_Share", index=False)
    monthly_trends.to_excel(writer, sheet_name="Monthly_Trends", index=False)
    day_night.to_excel(writer, sheet_name="Day_vs_Night", index=False)
    yearly.to_excel(writer, sheet_name="Yearly_Growth", index=False)


In [20]:
from google.colab import files
files.download("pharma_business_tables.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>